<a href="https://colab.research.google.com/github/norahmunoz/ds2002-fa26/blob/main/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one line item from an order**

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']

print("Unmatched vendors:")
print(unmatched[['vendor_id', 'qty']])

print("Units attached to unmatched vendors:", unmatched['qty'].sum())

joined

Unmatched vendors:
  vendor_id  qty
3      V-07    3
Units attached to unmatched vendors: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
vendor_totals = (
    joined.groupby('name', dropna=False)['qty']
    .sum()
    .reset_index()
)

# dropna=False keeps an unmatched vendor visible as NaN instead of silently removing it.
vendor_totals

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = (
    joined.groupby('zone', dropna=False)['qty']
    .sum()
    .reset_index()
)

print(zone_totals)
print("Total quantity represented by zone:", zone_totals['qty'].sum())

  zone  qty
0    A    4
1  NaN    3
Total quantity represented by zone: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

ragged_with_lines = [x for x in ragged_json if 'lines' in x]

ragged_lines = pd.json_normalize(
    ragged_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** One row per line item makes it easier to see what was sold and how much of each item was sold. If each row was a whole order, it would be harder to analyze the individual items.

**b)** A missing quantity means we don't know how many were sold, while zero means we know that none were sold. Treating missing values as zero could give us incorrect totals.
